# PB05 — Read come Pre-training / Augmentation per Img

**Ipotesi**: le epoche di lettura visiva (`_read`) contengono un segnale più robusto
dello stesso contenuto semantico. Usarle come pre-training o augmentation
può migliorare la decodifica dell'imagined speech.

**Strategie**:
1. **Augmentation**: train su img + read insieme, test su img
2. **Pre-training**: train encoder su read (più segnale), fine-tune su img
3. **Multi-task**: train su img e read con loss condivisa

Questo notebook implementa la strategia 1 (augmentation) come baseline.

In [ ]:
from pathlib import Path
import json

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists()),
    Path().resolve()
)

DATA_ROOT      = project_root / 'data' / 'raw_csv' / 'training_set'
CONFIGS        = project_root / 'configs' / 'label_schemes'
SFREQ = 256
N_CHAN = 61
N_SAMP = 384
CLUSTER_SCHEME = 'concr4'

# Split subject-independent (stesso di Paolo)
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

with open(CONFIGS / 'label2idx.json') as f:
    word2idx = json.load(f)
with open(CONFIGS / f'labelid2cluster_{CLUSTER_SCHEME}.json') as f:
    label2cluster = {int(k): v for k, v in json.load(f).items()}
N_CLASSES = len(set(label2cluster.values()))

print(f'Schema: {CLUSTER_SCHEME} | {N_CLASSES} classi | Chance: {1/N_CLASSES:.1%}')

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm
print('Import OK')

In [ ]:
# ============================================================
# DEVICE + limiti per macchina condivisa (spinlabs01)
#   - usa CUDA se disponibile (RTX 5090), altrimenti MPS/CPU
#   - cap VRAM così non si manda in OOM gli altri utenti
#   - oom_score_adj: in caso di OOM RAM, muore questo processo per primo
# ============================================================
import torch

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.cuda.set_per_process_memory_fraction(0.45, 0)   # ~14G dei 32 — il modello è piccolo, basta e avanza
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

try:
    with open('/proc/self/oom_score_adj', 'w') as _f:
        _f.write('800')
except Exception:
    pass  # non-Linux o permessi: ignora

print(f'Device: {DEVICE}')

In [ ]:
# ============================================================
# DATASET
# ============================================================

class EEGConditionDataset(Dataset):
    """Carica epoche di una condizione (_img o _read) per una lista di soggetti."""

    def __init__(self, subj_ids, data_root, condition='img', word2idx=None, label2cluster=None):
        self.samples = []
        for sid in subj_ids:
            for sess_dir in sorted(data_root.glob(f'P{sid:03d}_S*')):
                for f in sorted(sess_dir.glob(f'*_{condition}.csv')):
                    word = f.stem.replace(f'_{condition}', '')
                    if word not in word2idx:
                        continue
                    lid = word2idx[word]
                    if lid not in label2cluster:
                        continue
                    self.samples.append((str(f), label2cluster[lid]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        x = pd.read_csv(path, header=None).values.astype(np.float32)
        return torch.tensor(x).unsqueeze(0), label  # (1, 61, 384)


# Test dataset
ds_img  = EEGConditionDataset(SUBJ_TRAIN[:5], DATA_ROOT, 'img', word2idx, label2cluster)
ds_read = EEGConditionDataset(SUBJ_TRAIN[:5], DATA_ROOT, 'read', word2idx, label2cluster)
print(f'Train img: {len(ds_img)} | Train read: {len(ds_read)}')

In [ ]:
# ============================================================
# MODELLO: EEGNet semplificato
# ============================================================

class SimpleEEGNet(nn.Module):
    def __init__(self, n_classes, n_chan=61, n_samp=384):
        super().__init__()
        self.temporal = nn.Sequential(
            nn.Conv2d(1, 8, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(8, 16, (n_chan, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.25),
        )
        self.separable = nn.Sequential(
            nn.Conv2d(16, 16, (1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(0.25),
        )
        self.clf = nn.LazyLinear(n_classes)

    def forward(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        x = x.flatten(1)
        return self.clf(x)


# Test forward
model = SimpleEEGNet(N_CLASSES)
dummy = torch.zeros(4, 1, 61, 384)
out   = model(dummy)
print(f'Output shape: {out.shape}')


In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

def train_eval(train_ds, val_subjs, test_subjs, n_epochs=30, batch_size=64, lr=1e-3):
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                        num_workers=4, pin_memory=(DEVICE.type == 'cuda'))
    device = DEVICE
    model  = SimpleEEGNet(N_CLASSES).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(n_epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()

    model.eval()
    ds_test = EEGConditionDataset(test_subjs, DATA_ROOT, 'img', word2idx, label2cluster)
    loader_test = DataLoader(ds_test, batch_size=128, shuffle=False, num_workers=4)
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader_test:
            pred = model(x.to(device)).argmax(1).cpu().numpy()
            y_true.extend(y.numpy())
            y_pred.extend(pred)
    return balanced_accuracy_score(y_true, y_pred)


print('Training functions OK')


In [ ]:
# ============================================================
# ESPERIMENTO: img only vs img + read (augmentation)
# ============================================================

ds_train_img      = EEGConditionDataset(SUBJ_TRAIN, DATA_ROOT, 'img',  word2idx, label2cluster)
ds_train_read     = EEGConditionDataset(SUBJ_TRAIN, DATA_ROOT, 'read', word2idx, label2cluster)
ds_train_img_read = ConcatDataset([ds_train_img, ds_train_read])

print(f'Train img only:   {len(ds_train_img)}')
print(f'Train img + read: {len(ds_train_img_read)}')

print('\nTraining img only...')
bacc_img = train_eval(ds_train_img, SUBJ_VAL, SUBJ_TEST, n_epochs=30)
print(f'  Test bacc (img only): {bacc_img:.4f}')

print('\nTraining img + read...')
bacc_img_read = train_eval(ds_train_img_read, SUBJ_VAL, SUBJ_TEST, n_epochs=30)
print(f'  Test bacc (img+read): {bacc_img_read:.4f}')

print(f'\nΔ = {bacc_img_read - bacc_img:+.4f}  (chance={1/N_CLASSES:.4f})')